# Price sync

Fetches missing price history from Yahoo Finance for every symbol in the portfolio and updates `data/prices/`.
Each symbol gets its own cache file; only the missing date range is fetched on each run.

In [1]:
from datetime import date

from trades.brokers.ibkr import main
from trades.config import AppConfig
from trades.ledger import replay
from trades.market_data import prices

AS_OF_DATE = date.today()
config = AppConfig()

In [2]:
# Derive the symbol list and start date from the ledger
ledger = main.load_ledger(config)
result = replay.replay_ledger(ledger, config)

symbols = result.open_lots["symbol"].unique().sort().to_list()
first_trade_date = ledger["event_datetime"].min().date()
print(f"Symbols: {symbols}")
print(f"Fetching from {first_trade_date} to {AS_OF_DATE}")

Symbols: ['BND', 'QQQM', 'VOO', 'VXUS']
Fetching from 2026-01-26 to 2026-07-04


/Users/gabrielduguey/Documents/Perso/IBKR/src/trades/brokers/ibkr/main.py:71: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("meta").map_elements(json.loads)
with this one instead:
  + pl.col("meta").str.json_decode()

  return ledger.with_columns(pl.col("meta").map_elements(json.loads, return_dtype=pl.Object))


In [3]:
price_histories = prices.update_price_caches(symbols, since=first_trade_date, as_of=AS_OF_DATE, config=config)

for symbol, hist in price_histories.items():
    print(f"{symbol}: {len(hist)} trading days cached, latest close {hist['close'][-1]:.2f}")

BND: 110 trading days cached, latest close 73.11
QQQM: 110 trading days cached, latest close 293.42
VOO: 110 trading days cached, latest close 684.84
VXUS: 110 trading days cached, latest close 84.84


In [4]:
# Inspect one symbol's history
SYMBOL = symbols[0]
price_histories[SYMBOL]

price_date,close
date,f64
2026-01-26,74.339996
2026-01-27,74.279999
2026-01-28,74.25
2026-01-29,74.260002
2026-01-30,74.230003
…,…
2026-06-26,73.669998
2026-06-29,73.709999
2026-06-30,73.410004
